In [ ]:
# import os

# path = "/Users/ivanmkc/code/adk-samples/output/companies/blue_ridge_outfitters/run_0/Blue Ridge Outfitters_ _Gear Up for Your Adventure..md"
# with open(path, "r") as f:
#     text = f.read()

In [ ]:
# len(text.split("\n"))

In [ ]:
# len(text)

In [ ]:
from llm_auditor.claims import Claim, QueryAmbiguity

In [ ]:
import dspy
import mlflow
mlflow.dspy.autolog()
mlflow.set_experiment("extract_claims")

model_name = "gemini/gemini-2.5-pro-preview-06-05"
# model_name = "gemini/gemini-2.0-flash"
lm = dspy.LM(
    model=model_name,
    max_tokens=65535,
    # allowed_openai_params=["thinking"],
    # thinking={"type": "enabled", "budget_tokens": 1024},
)
dspy.configure(lm=lm)

In [ ]:
from typing import List, Optional, Iterator
import enum
import dspy
from pydantic import BaseModel, Field
import asyncio
from tqdm.asyncio import tqdm

# --- DSPy Signature for Query Generation ---
class QueryGenerationSignature(dspy.Signature):
    """
    Given a source text and a desired ambiguity level, generate relevant queries.

    - If 'straightforward', the question should be direct and factual, often starting with Who, What, When, or Where.
    - If 'ambiguous', the question should be more open-ended, subjective, or analytical, perhaps starting with Why, How, or asking for significance/implications.
    """

    source_text: str = dspy.InputField(
        desc="The full text from which to generate queries."
    )
    ambiguity: str = dspy.InputField(
        desc="The desired ambiguity level of the query. Must be either 'straightforward' or 'more ambiguous'."
    )
    generated_queries: List[str] = dspy.OutputField(
        desc="A list of generated queries."
    )


def batch_text(text: str, batch_size: int) -> Iterator[str]:
    """Yields batches of a given text by a specified number of lines."""
    batch = []
    
    for line in text.split("\n"):
        batch.append(line)

        if len(batch) >= batch_size:
            yield "\n".join(batch)
            batch = []
    
    if batch:
        yield "\n".join(batch)

# --- DSPy Module for Query Generation ---
class QueryGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.query_generator = dspy.ChainOfThought(QueryGenerationSignature)

    async def forward(self, source_text: str, ambiguity: QueryAmbiguity) -> dspy.Prediction:
        """
        Generates queries from the given source text based on the specified ambiguity.

        Args:
            source_text: The text to generate queries from.
            ambiguity: The desired ambiguity level ('straightforward' or 'more ambiguous').

        Returns:
            A dspy.Prediction object containing a list of generated queries.
        """

        semaphore = asyncio.Semaphore(5)
        query_generator_async = dspy.asyncify(self.query_generator)
        async def generate_queries(text: str) -> list[str]:
            async with semaphore:
                return await query_generator_async(source_text=text, ambiguity=ambiguity)
        
        generation_tasks = [generate_queries(batch)
                            for batch in batch_text(source_text, batch_size=4)
                            ]
        
        predictions = await tqdm.gather(*generation_tasks) 
        queries = [query
                        for prediction in predictions
                        for query in prediction.generated_queries
                       ]

        return queries

In [ ]:
# for batch in batch_text(text, 10):
#     print(batch)
#     print("---------")

In [ ]:
import dspy
# lm = dspy.LM(model="gemini/gemini-2.5-pro-preview-03-25")
lm = dspy.LM(
    model="gemini/gemini-2.5-pro-preview-03-25",
    max_tokens=65535,
    allowed_openai_params=['thinking'], 
    thinking={"type": "enabled", "budget_tokens": 2048},
)
dspy.configure(lm=lm)

In [ ]:
# extractor = dspy.ChainOfThought(ClaimExtractionSignature)
# extractor_async = dspy.asyncify(extractor)
# prediction = await extractor_async(source_text="Hello world.")

In [ ]:
# prediction.extracted_claims

In [ ]:
from typing import List, Tuple, Optional
import dspy
from pydantic import BaseModel, Field
import asyncio
from tqdm.asyncio import tqdm

# # --- Pydantic Model for a Claim and its Context ---
# class ContextAndClaim(BaseModel):
#     """A model representing a single claim and its corresponding context from the source text."""
    
#     context: str = Field(
#         ..., 
#         description="The sentences from the source text (copied verbatim) that support the claim's answer to a query."
#     )
#     claim: str = Field(
#         ...,
#         description="A self-contained claim that answers a query, either entailing or contradicting the context. Context-dependent words have been replaced with their explicit references."
#     )

# --- DSPy Signature for Answering a Query with a Claim ---
class ClaimGenerationSignature(dspy.Signature):
    """
    Given a source text and a specific query, generate an answer for the given query either correctly or incorrectly.
    The answer must be self-contained.
    Aim to create a distinct and significant answer.
    """

    source_text: str = dspy.InputField(
        desc="The full text containing the information to answer the query."
    )
    query: str = dspy.InputField(
        desc="The question that the generated claim must answer."
    )
    context: str = dspy.OutputField(
        description="The contiguous text from the source text (copied verbatim) for the correct and incorrect answers to the query."
    )
    answer_correct: str = dspy.OutputField(
        description="A self-contained correct answer to the query, entailing the context. Avoid ambiguity or speculation. Context-dependent words have been replaced with their explicit references."
    )
    answer_incorrect: str = dspy.OutputField(
        description="A self-contained incorrect answer to the query, contradicting the context. Context-dependent words have been replaced with their explicit references."
    )


# --- DSPy Module for Generating Claims from Queries ---
class ClaimGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.claim_generator = dspy.ChainOfThought(ClaimGenerationSignature)

    async def forward(self, queries: List[str], source_text: str) -> Tuple[List[str], List[str], List[str]]:
        """
        Generates claims that answer a list of queries, either correctly or incorrectly, based on a source text.

        Args:
            queries (List[str]): The list of questions to answer.
            source_text (str): The text to use for finding answers and context.

        Returns:
            A tuple containing two lists: the first is a list of contexts, and the second is a corresponding list of claims.
        """
 
        semaphore = asyncio.Semaphore(5)
        claim_generator_async = dspy.asyncify(self.claim_generator)
        
        async def generate_claim_for_query(query: str) -> Optional[tuple[str, str, str]]:
            """Asynchronously generates a single claim for a given query."""
            async with semaphore:
                prediction = await claim_generator_async(
                    query=query,
                    source_text=source_text,
                )
                if prediction:
                    return prediction.context, prediction.answer_correct, prediction.answer_incorrect
                return None

        # Create and run an asynchronous task for each query
        generation_tasks = [generate_claim_for_query(query) for query in queries]
        
        results: List[Optional[tuple[str, str, str]]] = await tqdm.gather(*generation_tasks) 
        
        # Unpack the results into separate lists of contexts and claims
        contexts = []
        correct_answers = []
        incorrect_answers = []
        for result in results:
            if result:
                context, correct_answer, incorrect_answer = result
                contexts.append(context)
                correct_answers.append(correct_answer)
                incorrect_answers.append(incorrect_answer)

        return contexts, correct_answers, incorrect_answers

In [ ]:
class ClaimRewriterSignature(dspy.Signature):
    """
    Given a source text and an input claim:
    1. Analyze the claim. If it's underspecified (e.g., uses pronouns like 'it', 'this', 'they' without clear antecedents, or refers to concepts vaguely), rewrite it to be self-contained and unambiguous by incorporating necessary context from the source text.
    2. The rewritten claim MUST be clearly and unambiguously either be supported by or contradicted by the source text.
    3. Provide a verdict ('supported' or 'contradicted') for the rewritten claim against the source text.
    """

    source_text: str = dspy.InputField(
        desc="The source text for context and verification."
    )
    claim: str = dspy.InputField(
        desc="The input claim to analyze and rewrite."
    )
    rewritten_claim: str = dspy.OutputField(
        desc="The rewritten, self-contained, and unambiguous claim."
    )
    is_supported: bool = dspy.OutputField(
        # Restricting to these two as per the problem statement's emphasis
        desc="Verdict for the rewritten_claim: True if 'supported' and False if 'contradicted' by the source_text."
    )
    reasoning: str = dspy.OutputField(
        desc="Brief reasoning for the verdict and any significant rewrites made to achieve clarity and verifiability."
    )

class ClaimRewriter(dspy.Module):
    def __init__(self):
        super().__init__()
        # Using Predict as ChainOfThought might be overkill if the prompt is strong enough,
        # but CoT is generally more robust for complex reasoning.
        self.rewriter_predictor = dspy.ChainOfThought(ClaimRewriterSignature)

    async def forward(self, claim: str, source_text: str) -> dspy.Prediction:
        """
        Rewrites a claim for clarity and verifies if it's supported or contradicted by the source text.

        Args:
            claim: The claim string to process.
            source_text: The source text to use for context and verification.

        Returns:
            A dspy.Prediction object containing 'rewritten_claim', 'verdict', and 'reasoning'.
        """
        # dspy.Predict/ChainOfThought are not async by default.
        # We need to use dspy.asyncify for them if we want to await their calls.
        rewriter_predictor_async = dspy.asyncify(self.rewriter_predictor) # batch_size=1 as we process one claim at a time here
        
        # If source_text is very long, the LLM might struggle.
        # However, for claim rewriting/verification, the relevant source context is often local.
        # The current FalsehoodExtractor already batches source_text for generation,
        # so the source_text passed here will be a manageable chunk.
        prediction = await rewriter_predictor_async(claim=claim, source_text=source_text)
        return prediction


In [ ]:
import pathlib
import glob
from yaml import SafeLoader, SafeDumper
import yaml
from dataclasses import asdict

query_extractor_module = QueryGenerator()
claim_generator_module = ClaimGenerator()
rewriter = ClaimRewriter()


class Cache:
    """
    A class to manage caching of processed data to disk using YAML.

    Attributes:
        cache_dir: The directory where cache files will be stored.
        _locks: A dictionary to store locks for each cache file, to avoid race conditions.
    """
    def __init__(self, cache_dir: pathlib.Path):
        """
        Initializes the cache.

        Args:
            cache_dir: The directory to store cache files.
        """
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self._locks: dict[pathlib.Path, asyncio.Lock] = {}

    def _get_cache_file_path(self, filepath: pathlib.Path) -> pathlib.Path:
        """
        Generates the cache file path for a given input file.

        Args:
            filepath: The path to the input file.

        Returns:
            The path to the corresponding cache file.
        """
        filename = filepath.name.replace('.', '_') + '.yaml'
        return self.cache_dir / filename

    async def get_cached_data(self, filepath: pathlib.Path) -> list[Claim] | None:
        """
        Retrieves cached data for a given file, if available.

        Args:
            filepath: The path to the input file.

        Returns:
            A list of ClaimWithMetadata objects if cached data is found, otherwise None.
        """
        cache_filepath = self._get_cache_file_path(filepath)
        if cache_filepath.exists():
            if cache_filepath not in self._locks:
                self._locks[cache_filepath] = asyncio.Lock()
            async with self._locks[cache_filepath]:
                try:
                    text = cache_filepath.read_text()
                    data = yaml.load(text, Loader=SafeLoader)
                    if data is None:
                        return None
                    return [Claim(**item) for item in data]
                except yaml.YAMLError:
                   print(f"Error decoding yaml cache file {cache_filepath}, will reprocess the file")
                   return None
        return None

    async def save_cached_data(self, filepath: pathlib.Path, data: list[Claim]):
        """
        Saves processed data to the cache for a given file.

        Args:
            filepath: The path to the input file.
            data: A list of ClaimWithMetadata objects to be cached.
        """
        cache_filepath = self._get_cache_file_path(filepath)
        if cache_filepath not in self._locks:
                self._locks[cache_filepath] = asyncio.Lock()
        async with self._locks[cache_filepath]:
            text = yaml.dump([asdict(item) for item in data], Dumper=SafeDumper)
            cache_filepath.write_text(text)



async def process_file(
        filepath: pathlib.Path, 
        semaphore: asyncio.Semaphore, 
        cache: Cache,
        ambiguity: QueryAmbiguity) -> list[Claim]:
    cached_data = await cache.get_cached_data(filepath)
    if cached_data:
        print(f"Cache hit for {filepath}")
        return cached_data

    print(f"Cache miss for {filepath}, processing file")
    # Open file
    with open(str(filepath), 'r') as f:
        text = f.read()

    async with semaphore:
        # extracted_claims = await claim_extractor_module.forward(
        #     source_text=text,

        queries = await query_extractor_module.forward(
            source_text=text,
            ambiguity=ambiguity
        )

        contexts, correct_answers, incorrect_answers = await claim_generator_module.forward(
            queries=queries,
            source_text=text,
        )

        all_contexts = contexts + contexts
        all_claims = correct_answers + incorrect_answers
        all_queries = queries + queries

        is_supported_all = [True for _ in range(len(correct_answers))] + [False for _ in range(len(incorrect_answers))]

        rewritten_predictions = await tqdm.gather(*[rewriter.forward(claim=claim, source_text=text) for claim in all_claims])

    # TODO: Verify context str exists in source
    source_contains_contexts = [context.strip().removesuffix("...").removeprefix("...") in text for context in all_contexts]

    result = [
            Claim(
                query=query,
                claim=claim,
                context=context,
                source_contains_context=source_contains_context,
                is_supported=is_supported,
                is_supported_after_rewriting=rewritten_prediction.is_supported,
                reasoning=rewritten_prediction.reasoning,
                source_file_path=str(filepath),
                ambiguity=ambiguity
            )
            for is_supported, query, context, source_contains_context, claim, rewritten_prediction in zip(
                is_supported_all,
                all_queries,
                all_contexts,
                source_contains_contexts,
                all_claims,
                rewritten_predictions)
        ]
    await cache.save_cached_data(filepath, result)
    return result


async def process_folder(
        dir_or_file_path: pathlib.Path, 
        cache_dir: pathlib.Path,
        ambiguity: QueryAmbiguity
        ) -> list[Claim]:

    files = [str(dir_or_file_path)]
    # Check if file is a folder
    if dir_or_file_path.is_dir():
        print(f"Looking recursively through {str(dir_or_file_path)} for .md files")
        # Get files recursively
        # Create the pattern to search for.
        # "**" matches any files and zero or more directories and subdirectories.
        # "*.md" matches any file ending with .md

        pattern = dir_or_file_path / "**" / "*.md"

        # Use glob.glob with recursive=True to find files recursively
        files = glob.glob(str(pattern), recursive=True)

    print(files)
    print(f"Found {len(files)} files")

    semaphore = asyncio.Semaphore(3)
    cache = Cache(cache_dir)

    claims_per_file = await tqdm.gather(*[
        process_file(
            filepath=pathlib.Path(file), 
            semaphore=semaphore, 
            cache=cache, 
            ambiguity=ambiguity
            ) 
            for file in files
        ])

    return [claim for claims in claims_per_file for claim in claims]
    
    

In [ ]:
claims = await process_folder(
        dir_or_file_path=pathlib.Path("/Users/ivanmkc/code/adk-samples/output/companies/blue_ridge_outfitters/run_0/"),
        # dir_or_file_path=pathlib.Path("/Users/ivanmkc/code/adk-samples/output/companies/blue_ridge_outfitters/run_0/Environmental Initiatives and Conservation Partnerships.md"),
        cache_dir=pathlib.Path("/Users/ivanmkc/code/adk-samples/output/claim_cache_with_queries"),
        ambiguity="straightforward"
)

In [ ]:
claims

In [ ]:
assert all([claim.is_supported == claim.is_supported_after_rewriting for claim in claims]), "Pre and post rewriting is_supported are not consistent"

In [ ]:
[claim for claim in claims if claim.is_supported != claim.is_supported_after_rewriting]


In [ ]:
[claim for claim in claims if not claim.source_contains_context]

In [ ]:
# # Create claims
# claims = []
# for (is_supported, prediction) in zip([True for _ in range(len(extracted_claims))] + [False for _ in range(len(false_claims))], rewritten_predictions):
#     claims.append(Claim(claim=prediction.rewritten_claim, is_supported=is_supported))


In [28]:
import yaml
from pathlib import Path
from dataclasses import asdict

claims_dir_path = Path("claims")
# os.mkdir(claims_dir_path)

# Write predictions
claims_as_dicts = [asdict(claim) for claim in claims]

with open(claims_dir_path / "claims_with_queries.yaml", 'w', encoding='utf-8') as f:
    yaml.dump(claims_as_dicts, f, default_flow_style=False, allow_unicode=True)

In [ ]:
# import yaml
# from pathlib import Path
# from dataclasses import asdict

# claims_dir_path = Path("claims")

# # Read the YAML file
# with open(claims_dir_path / "claims_with_queries.yaml", "r") as f:
#     claims_from_yaml = yaml.safe_load(f)

# # Convert the list of dictionaries back to a list of Claim objects
# reconstructed_claims_list = [Claim(**claim_data) for claim_data in claims_from_yaml]

In [33]:
type(reconstructed_claims_list[0].source_file_path)

str

In [ ]:
import pandas as pd

df = pd.DataFrame(claims_as_dicts)

In [ ]:
df['query'].value_counts()

In [ ]:
df.sort_values(['query']).head()

In [ ]:
for _, row in df.sort_values(['query', 'is_supported'])[:10].iterrows():
    print(f"Q: {row['query']}")
    print(f"Is correct?: {row['is_supported']}")
    print(f"A: {row['claim']}\n\n\n")

In [ ]:
# df[df['query'] == "What is the name of Blue Ridge Outfitters' private label brand?"].source_file_path.value_counts()